# Chapitre 5 - Cours 2 : journaux de réplication et retard des suiveurs

Le cours 1 a traité le journal de réplication comme une boîte noire. Ce cours l'ouvre
(que contient-il exactement ?), puis étudie la conséquence la plus visible de la
réplication asynchrone : des suiveurs **en retard**, qui font voir aux utilisateurs des
choses étranges. Il couvre les sections *Implementation of Replication Logs* et
*Problems with Replication Lag* (*Reading Your Own Writes*, *Monotonic Reads*,
*Consistent Prefix Reads*, *Solutions for Replication Lag*).

**Plan.**

1. Réplication par instructions, et pourquoi elle diverge.
2. Expédition du journal d'écriture anticipée (WAL).
3. Réplication logique, par lignes.
4. Le retard de réplication et la cohérence à terme.
5. Lire ses propres écritures.
6. Lectures monotones.
7. Lectures à préfixe cohérent.
8. Choisir une parade.

In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import replication as rp
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)

## 1. Réplication par instructions (section *Statement-based replication*)

L'idée la plus simple : le leader journalise chaque **instruction** d'écriture
(`INSERT`, `UPDATE`, `DELETE`) et chaque suiveur la ré-exécute comme si un client la lui
avait envoyée. C'est compact, mais cela casse dès qu'une instruction n'est pas
**déterministe** :

- une fonction comme `NOW()` ou `RAND()` donne une valeur différente sur chaque réplique ;
- une instruction qui dépend des données existantes (`UPDATE ... WHERE ...`, une colonne
  auto-incrémentée) doit être exécutée **exactement dans le même ordre** partout, ce qui
  gêne avec des transactions concurrentes ;
- les effets de bord (déclencheurs, procédures stockées, fonctions utilisateur) doivent
  être parfaitement déterministes.

Rejouons-le avec la mini-base `rp.MiniDB` : chaque réplique a sa propre horloge (le
suiveur avance de 0,7 s) et son propre générateur aléatoire. Trois instructions : une
insertion avec `NOW()` et `RAND()`, puis deux transactions concurrentes sur le même
compte (doubler le solde, ajouter 10). Le suiveur reçoit les deux transactions dans
l'ordre inverse, ce qui arrive quand elles se chevauchent sur le leader.

In [2]:
STMTS_SQL = ["INSERT INTO evt (at, tirage) VALUES (NOW(), RAND())",
             "UPDATE comptes SET solde = solde * 2",
             "UPDATE comptes SET solde = solde + 10"]
stmts = [lambda db, t: db.insert("evt", {"at": db.now(t), "tirage": db.rand()}),
         lambda db, t: [db.update("comptes", r["id"], {"solde": r["solde"] * 2}) for r in db.select("comptes")],
         lambda db, t: [db.update("comptes", r["id"], {"solde": r["solde"] + 10}) for r in db.select("comptes")]]

def fresh(name, offset, seed):
    db = rp.MiniDB(name, clock_offset=offset, seed=seed)
    db.insert("comptes", {"id": 1, "solde": 100})
    db.changes.clear()
    return db

def compare(L, F):
    rows = []
    for table in ("evt", "comptes"):
        for pk, row in L.tables[table].items():
            for col, v in row.items():
                if col != "id":
                    w = F.tables[table][pk][col]
                    rows.append({"table": table, "colonne": col, "leader": v, "suiveur": w,
                                 "identique": "oui" if v == w else "NON"})
    return pd.DataFrame(rows)

L, F = fresh("leader", 0.0, 1), fresh("suiveur", 0.7, 2)
rp.replicate_statements(stmts, L, [F], times=[1.0, 2.0, 2.0], follower_order=[[0, 2, 1]])
compare(L, F)

,table,colonne,leader,suiveur,identique
0,evt,at,1.000,1.750,NON
1,evt,tirage,0.134,0.956,NON
2,comptes,solde,210.000,220.000,NON


Les trois valeurs divergent : l'horodatage (horloges différentes), le tirage (générateurs
différents) et le solde, $2 \times 100 + 10 = 210$ sur le leader contre
$2 \times (100 + 10) = 220$ sur le suiveur. Personne n'a signalé d'erreur : les répliques
ont **silencieusement** cessé d'être des copies.

Les parades existent : le leader peut remplacer chaque appel non déterministe par sa
valeur au moment de journaliser l'instruction, et imposer un ordre d'exécution total.
Exécuter des instructions déterministes dans un ordre fixé, c'est la **réplication de
machine à états**. MySQL utilisait la réplication
par instructions avant la version 5.1 et bascule aujourd'hui vers la réplication par
lignes dès qu'une instruction n'est pas déterministe ; VoltDB la rend sûre en exigeant
des transactions déterministes. Mais le déterminisme est difficile à garantir, d'où les
deux autres méthodes.

> **À retenir.** Rejouer les instructions SQL sur chaque réplique est compact, mais toute
> source de non-déterminisme (horloge, hasard, ordre, effets de bord) fait diverger les
> copies sans bruit.

## 2. Expédition du journal d'écriture anticipée (section *Write-ahead log shipping*)

Le moteur de stockage tient déjà un journal : le **WAL** (chapitre 3), où
toute modification est écrite avant d'être appliquée aux pages, pour survivre aux
pannes. Puisque ce journal suffit à reconstruire l'état exact des fichiers, il suffit
de l'envoyer aussi aux suiveurs, qui reconstruisent des fichiers **identiques octet pour
octet**. C'est ce que font PostgreSQL et Oracle.

L'inconvénient : le WAL décrit les données à très bas niveau (« tels octets dans tel bloc
du disque »). La réplication est donc **couplée au format de stockage**. Illustrons-le
avec un enregistrement de WAL « physique » qui écrit un solde dans une page, interprété
par une version du moteur qui a modifié la disposition des pages (un en-tête de ligne
passé de 4 à 8 octets).

In [3]:
import struct

def page_v1(solde):        # en-tête de ligne 4 octets (id), puis solde sur 4 octets
    return bytearray(struct.pack("<ii", 1, solde)) + bytearray(8)

def lire_v1(page):
    return struct.unpack_from("<i", page, 4)[0]

def lire_v2(page):         # version suivante : en-tête de 8 octets (id + drapeaux)
    return struct.unpack_from("<i", page, 8)[0]

wal_record = {"page": 17, "offset": 4, "octets": struct.pack("<i", 210)}   # « écrire 210 à l'octet 4 »
p_leader, p_suiveur = page_v1(100), page_v1(100)
for p in (p_leader, p_suiveur):
    p[wal_record["offset"]:wal_record["offset"] + 4] = wal_record["octets"]
pd.DataFrame({"version du moteur qui lit la page": ["v1 (même version que le leader)", "v2 (moteur mis à jour)"],
              "solde lu": [lire_v1(p_suiveur), lire_v2(p_suiveur)]})

,version du moteur qui lit la page,solde lu
0,v1 (même version que le leader),210
1,v2 (moteur mis à jour),0


Le même enregistrement, appliqué fidèlement, donne un solde de 210 pour un moteur v1 et
une valeur absurde (0) pour un moteur v2 qui cherche le solde 4 octets plus loin. C'est
pourquoi, avec l'expédition du WAL, leader et suiveurs doivent en général tourner
**exactement** la même version du logiciel. Conséquence opérationnelle majeure : pas de
mise à jour sans interruption. Avec un protocole qui tolère un suiveur plus récent, on
mettrait d'abord à jour les suiveurs, puis on basculerait vers l'un d'eux ; avec le WAL,
il faut arrêter le service.

> **À retenir.** Expédier le WAL donne des copies exactes au moindre coût, mais lie la
> réplication au format physique du moteur : versions identiques obligatoires, mises à
> jour avec interruption.